# heapq: The Why (not just the How)

Basic examples show you push/pop. That's useless for understanding **when** to use it.

The insight: heapq is not a data structure you dump data into. It's a tool for **maintaining order lazily** — you only sort what you need, when you need it.

Here are the real patterns.

## Pattern 1: Top-K (the one you'll actually use most)

In [5]:
import heapq

# Problem: find the 3 largest numbers in a huge stream.
# Naive: collect all, sort, slice → O(n log n) memory & time.
# Heap: keep a min-heap of size 3 → O(n log k) where k=3, memory = O(k).

stream = [42, 1, 99, 7, 55, 100, 23, 88, 3]

# Pattern: push onto min-heap, if heap > k, pop smallest (which we don't care about)
top3 = []
for val in stream:
    heapq.heappush(top3, val)
    if len(top3) > 3:
        heapq.heappop(top3)  # discard the smallest, it's not in our top 3

print(f"Top 3: {sorted(top3, reverse=True)}")  # heap itself is NOT sorted, only heap[0] is min
print(f"Raw heap: {top3}")

Top 3: [100, 99, 88]
Raw heap: [88, 100, 99]


In [6]:
# Same thing, but heapq provides it built-in:
print(heapq.nlargest(3, stream))
print(heapq.nsmallest(3, stream))

[100, 99, 88]
[1, 3, 7]


In [7]:
# Real example: track top 5 most expensive orders from a live feed
import random

orders = [{"id": i, "amount": random.randint(100, 10000) / 100} for i in range(100)]

# Use key= to extract comparison value
top5 = heapq.nlargest(5, orders, key=lambda o: o["amount"])
for o in top5:
    print(f"  Order #{o['id']}: ${o['amount']:.2f}")

  Order #94: $99.99
  Order #35: $97.88
  Order #37: $96.42
  Order #16: $94.28
  Order #6: $92.97


## Pattern 2: Merge sorted streams without loading everything into memory

In [8]:
# Imagine 3 sorted log files, each 100GB. You can't load them all.
# heapq.merge() reads lazily — it only keeps one entry per file in memory.

file_a = [1, 4, 7, 10]
file_b = [2, 5, 8, 11]
file_c = [0, 3, 6, 9]

merged = heapq.merge(file_a, file_b, file_c)
print(f"Merged: {list(merged)}")

# This is how external sort works: sort chunks → write to files → merge via heap.
# Each call to next() pulls the smallest element from the right source.

Merged: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11]


In [9]:
# merge() also supports key= (e.g. merging log entries by timestamp)

import datetime

# ISO 8601 timestamps → lexicographic sort = chronological. No key needed.
logs_a = ["2024-01-05 10:00 login", "2024-01-05 10:05 logout"]
logs_b = ["2024-01-05 10:02 purchase", "2024-01-05 10:07 refund"]
print(list(heapq.merge(logs_a, logs_b)))  # works because strings sort correctly

# Non-ISO timestamps → need key= to parse first
logs_c = ["Jan 5 10:00 login",   "Jan 5 10:05 logout"]
logs_d = ["Jan 5 10:02 purchase", "Jan 5 10:07 refund"]
print(list(heapq.merge(logs_c, logs_d,
    key=lambda line: datetime.datetime.strptime(line[:15], "%b %d %H:%M"))))

[(100, 'login'), (200, 'purchase'), (300, 'logout'), (400, 'refund')]


## Pattern 3: Running median (why it's cool)

In [10]:
# You want the median after each new number arrives.
# Naive re-sort every time: O(n^2 log n).
# Two heaps: O(n log n) total.
#
# One max-heap for the lower half, one min-heap for the upper half.
# Keep them balanced. Median is at the boundary between them.
#
# Why does LO store negatives? Because heapq has no max-heap.
# lo = max-heap for lower half (want the LARGEST there).
#   Trick: store -x. -9 is "smaller" than -5 in min-heap,
#   so lo[0] = most negative = negated max of lower half.
# hi = min-heap for upper half. No negation needed.
#   hi[0] = smallest of upper half.

def running_median(iterable):
    lo = []  # lower half, stored negated (behaves as max-heap)
    hi = []  # upper half, normal min-heap

    for x in iterable:
        if len(lo) == len(hi):
            # Push to hi first, then move hi's smallest to lo
            heapq.heappush(lo, -heapq.heappushpop(hi, x))
            yield -lo[0]
        else:
            # lo is larger, push to lo first, then move lo's largest to hi
            heapq.heappush(hi, -heapq.heappushpop(lo, -x))
            yield (-lo[0] + hi[0]) / 2

data = [5, 9, 4, 12, 8, 9]
print(list(running_median(data)))
# Output: [5, 7, 5, 7, 8, 8.5]
# After 5:   median = 5
# After 5,9: median = (5+9)/2 = 7
# After 5,9,4: sorted = [4,5,9], median = 5
# ... etc

[5, 7.0, 5, 7.0, 8, 8.5]


## Pattern 4: Priority queue with update/delete (the full version from docs)

In [11]:
# The problem: you have tasks with changing priorities.
# You can't just find and update an entry in a heap — that breaks invariants.
# Solution: mark stale entries as REMOVED, add new ones. Lazy cleanup on pop.

import itertools

REMOVED = "<removed>"

class PriorityQueue:
    def __init__(self):
        self._pq = []                    # [(priority, counter, task), ...]
        self._entry_finder = {}          # task → [priority, counter, task]
        self._counter = itertools.count() # tie-breaker for equal priorities

    def add(self, task, priority=0):
        if task in self._entry_finder:
            self.remove(task)
        count = next(self._counter)
        entry = [priority, count, task]
        self._entry_finder[task] = entry
        heapq.heappush(self._pq, entry)

    def remove(self, task):
        entry = self._entry_finder.pop(task)
        entry[-1] = REMOVED  # mark stale, don't actually delete from heap

    def pop(self):
        while self._pq:
            priority, count, task = heapq.heappop(self._pq)
            if task is not REMOVED:
                del self._entry_finder[task]
                return task, priority
        raise KeyError("empty")

    def peek(self):
        while self._pq and self._pq[0][-1] is REMOVED:
            heapq.heappop(self._pq)
        if self._pq:
            return self._pq[0][-1], self._pq[0][0]
        raise KeyError("empty")

In [12]:
pq = PriorityQueue()
pq.add("write docs", priority=3)
pq.add("fix bug", priority=1)
pq.add("refactor", priority=2)

print(f"Next: {pq.pop()}")  # ('fix bug', 1)

# Oh wait, docs became urgent!
pq.add("write docs", priority=0)  # updates existing entry
print(f"Next: {pq.pop()}")  # ('write docs', 0)
print(f"Next: {pq.pop()}")  # ('refactor', 2)

Next: ('fix bug', 1)
Next: ('write docs', 0)
Next: ('refactor', 2)


## Pattern 5: Simulation / Event scheduler

In [13]:
# A heap IS a scheduler. The "key" is time. Pop = next event to fire.
# Events can schedule new events (with future timestamps).

def simulate():
    events = []  # (time, callback, arg)
    now = 0

    def schedule(delay, callback, arg=None):
        heapq.heappush(events, (now + delay, callback, arg))

    def log(msg):
        print(f"  t={now:3d}: {msg}")

    # Schedule initial events
    schedule(5, log, "Server started")
    schedule(2, log, "DB connected")
    schedule(8, log, "Cache warmed")
    schedule(1, log, "Config loaded")

    while events:
        now, callback, arg = heapq.heappop(events)
        callback(arg)

    print(f"\nFinal time: {now}")

simulate()

  t=  1: Config loaded
  t=  2: DB connected
  t=  5: Server started
  t=  8: Cache warmed

Final time: 8


In [ ]:
# Event that schedules another event:
# BUG FIX: store (time, counter, callback, arg) so events at
# the same time don't try to compare callbacks (functions aren't < comparable).

import itertools  # noqa

def simulate_with_chain():
    counter = itertools.count()
    events = []  # (time, counter, callback, arg)
    now = 0

    def schedule(delay, callback, arg=None):
        heapq.heappush(events, (now + delay, next(counter), callback, arg))

    def process_request(req_id):
        nonlocal now
        print(f"  t={now:3d}: Processing request {req_id}")
        schedule(3, send_response, req_id)  # schedules future work

    def send_response(req_id):
        print(f"  t={now:3d}: Sent response for request {req_id}")

    schedule(1, process_request, 1)
    schedule(4, process_request, 2)

    while events:
        now, _, callback, arg = heapq.heappop(events)
        callback(arg)

simulate_with_chain()

  t=  1: Processing request 1
  t=  4: Processing request 2
  t=  4: Sent response for request 1
  t=  7: Sent response for request 2


## Pattern 6: Fixed-size LRU-like eviction by score

In [15]:
# Keep only the 5 best scores seen so far.
# Push everything, pop the worst when over capacity.

best_scores = []
MAX_SIZE = 5

scores = [67, 45, 92, 88, 73, 55, 99, 40, 81, 76]

for s in scores:
    heapq.heappush(best_scores, s)
    if len(best_scores) > MAX_SIZE:
        discarded = heapq.heappop(best_scores)
        print(f"  Score {s:3d}: heap full, evicted {discarded}")
    else:
        print(f"  Score {s:3d}: added")

print(f"\nFinal top {MAX_SIZE}: {sorted(best_scores, reverse=True)}")

  Score  67: added
  Score  45: added
  Score  92: added
  Score  88: added
  Score  73: added
  Score  55: heap full, evicted 45
  Score  99: heap full, evicted 55
  Score  40: heap full, evicted 40
  Score  81: heap full, evicted 67
  Score  76: heap full, evicted 73

Final top 5: [99, 92, 88, 81, 76]


## Pattern 7: Max-heap

heapq is min-heap by default. For max-heap you have two options:

In [16]:
# Python < 3.14: Negate your values
# Store -x, so the "smallest" (most negative) is actually the largest x

import heapq

data = [3, 1, 4, 1, 5, 9]

# Build max-heap by negating
max_heap = [-x for x in data]
heapq.heapify(max_heap)

# Pop largest values
while max_heap:
    val = -heapq.heappop(max_heap)  # negate back
    print(val, end=" ")  # 9 5 4 3 1 1
print()

9 5 4 3 1 1 


In [17]:
# The negation trick works for all heap operations:

h = []
heapq.heappush(h, -5)   # push 5 as max-heap
heapq.heappush(h, -2)   # push 2
heapq.heappush(h, -8)   # push 8

print(-heapq.heappop(h))  # 8 (largest)
print(-heapq.heappop(h))  # 5
print(-heapq.heappop(h))  # 2

8
5
2


**The negation trick is foolproof** — works on any Python version. Python 3.14+ adds `heapify_max`, `heappush_max`, `heappop_max` as syntactic sugar, but negating is simpler to remember.

## The mental model

| When you need... | Use this |
|---|---|
| Always access the **smallest** (or largest) element quickly | Heap |
| Only the **top K** from a huge dataset | `nlargest`/`nsmallest`, or manual size-bounded heap |
| Merge **multiple sorted streams** lazily | `heapq.merge()` |
| A **priority queue** with update/delete | Dictionary + lazy deletion pattern |
| **Event/simulation scheduler** | Heap of (time, event) tuples |
| **Running median** or any streaming statistic | Two heaps (lo max-heap, hi min-heap) |
| Just sort some data once | `sorted()` — don't overcomplicate it |

## Key gotchas

- **heappop() pops the SMALLEST.** Name says "pop from heap" — doesn't say which end. That's because heapq is always a **min-heap**: `heap[0]` IS the minimum by definition. `heappop` returns `heap[0]`, then reheapifies. Think of it as "pop the root" — the root is always the winner (the smallest).
- **Max-heap = negate.** Store `-x`, pop `-heappop(h)`. Every time. Don't overthink it.
- **The list is NOT sorted.** Only `heap[0]` is the minimum. Print the heap? It looks random.
- **Tuple comparison:** `(priority, task)` breaks if priorities tie and tasks aren't comparable. Use `(priority, counter, task)`.
- **heapify() is O(n)** — converting an existing list is cheaper than pushing one by one (O(n log n)).
- **heapreplace vs heappushpop:** `replace` always pops AND pushes. `pushpop` might just return the pushed item if it's smaller than heap[0].